In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import jax
import jax.numpy as jnp

from scipy.spatial import Delaunay

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# === Global Plotting Style ===
plt.style.use('tableau-colorblind10')
plt.rcParams.update({
    'text.usetex': True,
    'font.size': 16,
    'axes.linewidth': 1.2,
    'axes.edgecolor': 'black',
    'xtick.major.size': 5,
    'ytick.major.size': 5
})

import config.config as config

import plotting_tools.basic_plotting_tools as plot_tools
import plotting_tools.plot_manifolds as plot_manifolds
import plotting_tools.plot_charts as plot_charts

import differential_geometry.aux_tools as aux

import differential_geometry.manifolds as manifolds
import differential_geometry.charts as charts
import differential_geometry.functions as functions
import differential_geometry.curves as curves
import differential_geometry.fields as fields

In [ ]:
# === Define a manifold ==============================
manifold = manifolds.Manifold(
    param_dim=2, ambient_dim=3,
    embedding_func=aux.surface_embedding_factory(
        amplitude=1.0, center=(0.0, 0.0), sigma=(jnp.pi, jnp.pi)
    ),
)

# === Define Charts ==================================
center_U = jnp.array([0.1, -0.7])
axes_U = (0.9, 1.5)
lambdas = jnp.linspace(0, 2 * jnp.pi, 600)
boundary_U = aux.boundary_ellipse_in_param_space(lambdas, center=center_U, axes=axes_U)
chart_U = charts.Chart(
    name="U",
    manifold=manifold,
    boundary=np.array(boundary_U),
    chart_map=aux.cartesian_chart_map,
)

center_V = jnp.array([0., 0.])
axes_V = (1.0, 1.0)
# center_V = jnp.array([0.7, 0.3])
# axes_V = (1.4, 0.9)
x_center_polar, y_center_polar = -0., -0.
# x_center_polar, y_center_polar = -1., -1.
lambdas = jnp.linspace(0, 2 * jnp.pi, 600)
boundary_V = aux.boundary_ellipse_in_param_space(lambdas, center=center_V, axes=axes_V)
chart_V = charts.Chart(
    name="V",
    manifold=manifold,
    boundary= np.array(boundary_V),
    chart_map=lambda params: aux.polar_chart_map(params, center=(x_center_polar, y_center_polar))
)

boundary_U_cap_V = charts.compute_chart_intersection(boundary_U, boundary_V)

chart_U_cap_V_X = charts.Chart(
    name="U_cap_V_X",
    manifold=manifold,
    boundary=np.array(boundary_U_cap_V),
    chart_map=aux.cartesian_chart_map,
)

chart_U_cap_V_Y = charts.Chart(
    name="U_cap_V_Y",
    manifold=manifold,
    boundary=np.array(boundary_U_cap_V),
    chart_map=lambda params: aux.polar_chart_map(params, center=(x_center_polar, y_center_polar)),
)

# === Define a point p ===============================
p_x, p_y = 0.15, 0.15
p_param = jnp.array([p_x, p_y])
p_in_M = manifold.embed(p_param)

# === Define curves ==================================
curve_gamma = curves.Curve(
    manifold=manifold, parametric_function=aux.wiggly_curve_factory(
        p_x, p_y, amplitude=jnp.pi/6, frequency=jnp.pi/1.4
    )
)
curve_delta = curves.Curve(
    manifold=manifold, parametric_function=aux.arching_curve_factory(
        p_x, p_y, amplitude=jnp.pi/6, frequency=jnp.pi/3
    )
)
# curve_gamma = curves.Curve(
#     manifold=manifold, parametric_function=aux.vertical_line_curve_factory(p_x, p_y)
# )
# curve_delta = curves.Curve(
#     manifold=surface_manifold, parametric_function=aux.circle_curve_factory(p_x, p_y)
# )

# === Define functions ===================================
def my_function(params):
    x, y = params
    return jnp.sqrt(x**2 + y**2)

function_f = functions.Function(
    manifold=manifold,
    function_parametric=my_function
)
function_f = None

def function_g_parametric(params):
    """
    Function that returns the z-coordinate of the manifold embedding.
    """
    z = manifold.embed(params)[..., 2]  # Take the third component (z)
    return z

function_g = functions.Function(
    manifold=manifold,
    function_parametric=function_g_parametric
)
function_g = None

# === Define Fields ===================================
field_X = fields.Field(manifold=manifold, vector_function=aux.circular_vector_field_parametric)
field_Y = fields.Field(manifold=manifold, vector_function=aux.constant_direction_vector_field_parametric(
        direction=(0.0, 1.0)
    )
)

In [ ]:
xmin, xmax = jnp.array([-jnp.pi, jnp.pi]) * jnp.pi/3
ymin, ymax = jnp.array([-jnp.pi, jnp.pi]) * jnp.pi/3
grid_resolution_field = 16

name_save_prefix = "TstarpM"

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
plot_manifolds.plot_manifold(
    ax=ax, manifold=manifold, resolution=100, xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax,
    color="skyblue", alpha=0.2, edgecolor="none", label=r"$\left(\mathcal{M}, \mathcal{O}, \mathcal{A}\right)_d$",
    label_position="top", label_fontsize=20, function=None, cmap="viridis", label_function="Scalar field $f$",
)

p_xyz = p_in_M[0] if p_in_M.shape[0] == 1 else p_in_M
ax.scatter(*p_xyz, color="k", s=60)

param_pts = jnp.array([[p_x, p_y]])
embedded = manifold.embed(param_pts)
jacobians = manifold.derivatives_at_params(param_pts)
labels = [r"$T_p\mathcal{M}$"]
plot_manifolds.plot_tangent_spaces(
    ax=ax, embedded_points=embedded, jacobians=jacobians, size=1.6, color="purple",
    alpha=0.2, resolution=10, line_width=2.0, labels=labels, label_fontsize=20,  label_offset=0.1
)
embedded += jnp.array([0.0, 0.0, 1.2])
labels = [r"$T_p^*\mathcal{M}$"]
plot_manifolds.plot_tangent_spaces(
    ax=ax, embedded_points=embedded, jacobians=jacobians, size=1.6, color="darkorange",
    alpha=0.15, resolution=10, line_width=2.0, labels=labels, label_fontsize=20,  label_offset=0.1
)

plot_charts.plot_chart_region_manifold(
    ax=ax, surface_manifold=manifold, chart=chart_U,
    color="red", n_points=512, linestyle="dashed", linewidth=1.5,
    fill_surface=True, alpha=0.15, edgecolor=None, linewidth_fill=0.5,
    label=r"$\mathcal{U}$", label_position="bottom", label_fontsize=14,
    draw_endpoints=True, endpoint_color="black", endpoint_size=30,
)
plot_charts.plot_chart_region_manifold(
    ax=ax, surface_manifold=manifold, chart=chart_V,
    color="blue", n_points=2**14, linestyle="dashed", linewidth=1.5,
    fill_surface=True, alpha=0.15, edgecolor=None, linewidth_fill=0.5,
    label=r"$\mathcal{V}$", label_position="bottom", label_fontsize=14,
    draw_endpoints=True, endpoint_color="black", endpoint_size=30,
)
plot_charts.plot_chart_region_manifold(
    ax=ax, surface_manifold=manifold, chart=chart_U_cap_V_X,
    color="purple", n_points=2**8, linestyle="dotted", linewidth=3.5,
    fill_surface=True, alpha=0.3, edgecolor=None, linewidth_fill=0.5,
    label=r"$\mathcal{U} \cap \mathcal{V}$", label_position="bottom", label_fontsize=14,
    draw_endpoints=True, endpoint_color="purple", endpoint_size=30,
)

plot_tools.plot_curve_on_manifold(
    ax=ax, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi),
    color="gold", linewidth=2.5,
    label=r"$\gamma$", label_position="start", label_fontsize=20, label_offset=(0.1, 0.1, 0.3)
)
plot_tools.plot_curve_on_manifold(
    ax=ax, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi),
    color="limegreen", linewidth=2.5,
    label=r"$\delta$", label_position="end", label_fontsize=20, label_offset=(0.1, 0.1, 0.3)
)

# plot_tools.plot_tangent_vectors_on_manifold(
#     ax=ax, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi),
#     step=30, scale=0.5, method="finite_difference",
#     style=dict(mutation_scale=1.5, arrowstyle="->,head_length=3.,head_width=2.", lw=2.0, color="gold")
# )
# plot_tools.plot_tangent_vectors_on_manifold(
#     ax=ax, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi),
#     step=30, scale=0.5, method="finite_difference",
#     style=dict(mutation_scale=1.5, arrowstyle="->,head_length=3.,head_width=2.", lw=2.0, color="limegreen")
# )

# plot_tools.plot_colored_curve_directional_derivative(
#     ax=ax, curve=curve_gamma, scalar_function=function_f, lambda_range=(-jnp.pi, jnp.pi), cmap="PiYG",
#     linewidth=3.0, label=None, label_position="center", inset_colorbar=True, inset_position=(0.01, 0.85),
#     inset_size=(2., 0.2), colorbar_orientation="horizontal", label_colorbar=r"$\left(\gamma \circ f\right)'$",
#     label_colorbar_position="top", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black"
# )
# plot_tools.plot_colored_curve_directional_derivative(
#     ax=ax, curve=curve_delta, scalar_function=function_f, lambda_range=(-jnp.pi, jnp.pi), cmap="bwr",
#     linewidth=3.0, label=None, label_position="center", inset_colorbar=True, inset_position=(0.68, 0.85),
#     inset_size=(2., 0.2), colorbar_orientation="horizontal", label_colorbar=r"$\left(\delta \circ f\right)'$",
#     label_colorbar_position="top", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black"
# )

# plot_tools.plot_vector_field_on_manifold(
#     ax, field=field_X, grid_resolution=grid_resolution_field, xlim=(xmin, xmax), ylim=(ymin, ymax),
#     vector_scale=0.3, colormap="YlOrBr", label=r"$X$", inset_position=(0.1, 0.7), inset_size=(0.6, 0.2), inset_orientation="horizontal"
# )
# plot_tools.plot_vector_field_on_manifold(
#     ax, field=field_Y, grid_resolution=grid_resolution_field, xlim=(xmin, xmax), ylim=(ymin, ymax),
#     vector_scale=0.3, colormap="Greens", label=r"$Y$", inset_position=(0.8, 0.7), inset_size=(0.6, 0.2), inset_orientation="horizontal"
# )

ax.view_init(elev=40.0, azim=-60.0)
plt.tight_layout()
fig.savefig(os.path.join(config.save_dir, name_save_prefix + "_manifold.pdf"))
plt.show()


# === Chart U plot ===
fig, ax = plt.subplots(figsize=(7, 7))

p_coords = chart_U.map_to_chart(p_param)
p_coords = p_coords[0] if p_coords.ndim == 2 and p_coords.shape[0] == 1 else p_coords
ax.scatter(*p_coords, color="black", s=60, zorder=5, label=r"$p$")

plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_U, param_space_data=chart_U.boundary_curve, color="red", linestyle="dashed",
    linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$x(\mathcal{U})$", label_position="bottom",
    label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=False, boundary_marker_color="red", boundary_marker_size=8
)
plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_U, param_space_data=chart_V.boundary_curve, color="blue", linestyle="dashed",
    linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$x(\mathcal{V})$", label_position="top",
    label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=False, boundary_marker_color="blue", boundary_marker_size=8
)
plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_U, param_space_data=chart_U_cap_V_X.boundary_curve, color="purple", linestyle="dotted",
    linewidth=3.5, alpha=0.3, edgecolor="none", label=r"$x(\mathcal{U} \cap \mathcal{V})$", label_position="center",
    label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=False, boundary_marker_color="purple", boundary_marker_size=8
)

plot_tools.plot_curve_in_chart(
    ax, chart=chart_U, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
    color="gold", linewidth=2.0, label=r"$\gamma$", label_position="end", label_fontsize=20, label_offset=(0.2, -0.2)
)
plot_tools.plot_curve_in_chart(
    ax, chart=chart_U, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
    color="limegreen", linewidth=2.0, label=r"$\delta$", label_position="start", label_fontsize=20, label_offset=(0.2, 0.2)
)

# plot_tools.plot_chart_components_of_curve_tangent(
#     ax, chart=chart_U, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
#     step=30, scale=0.3, color="gold", linewidth=1.8, head_width=0.1, head_length=0.2, zorder=5, method="finite_difference"
# )
# plot_tools.plot_chart_components_of_curve_tangent(
#     ax, chart=chart_U, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
#     step=30, scale=0.3, color="limegreen", linewidth=1.8, head_width=0.1, head_length=0.2, zorder=5, method="finite_difference"
# )

# plot_tools.plot_function_in_chart(
#     ax=ax, chart=chart_U, function=function_f, param_sampling_bounds=((xmin, xmax), (ymin, ymax)),
#     chart_xlim=(xmin, xmax), chart_ylim=(ymin, ymax), resolution=96, cmap="viridis", alpha=0.5, label_function="Scalar field $f$"
# )

# plot_tools.plot_colored_curve_directional_derivative_in_chart(
#     ax=ax, chart=chart_U, curve=curve_gamma, scalar_function=function_f, cmap="PiYG", label=r"$\gamma$", label_position="end",
#     inset_colorbar=True, inset_position=(0.02, 0.05), inset_size=(0.6, 0.2), colorbar_orientation="horizontal",
#     label_colorbar=r"$\left(\gamma \circ f\right)'$", label_colorbar_position="top"
# )

# plot_tools.plot_colored_curve_directional_derivative_in_chart(
#     ax=ax, chart=chart_U, curve=curve_delta, scalar_function=function_f, cmap="bwr", label=r"$\delta$", label_position="start",
#     inset_colorbar=True, inset_position=(0.85, 0.05), inset_size=(0.6, 0.2), colorbar_orientation="horizontal",
#     label_colorbar=r"$\left(\delta \circ f\right)'$", label_colorbar_position="top"
# )

# xv, yv = jnp.linspace(xmin, xmax, grid_resolution_field), jnp.linspace(ymin, ymax, grid_resolution_field)
# param_points = jnp.stack(jnp.meshgrid(xv, yv), axis=-1).reshape(-1, 2)
# points_in_chart = chart_U.map_to_chart(param_points)
# uX = field_X.evaluate_in_param_space(param_points)
# uY = field_Y.evaluate_in_param_space(param_points)
# def chart_map_fn(param):
#     return chart_U.map_to_chart(param)
# jac_chart = jax.vmap(jax.jacrev(chart_map_fn))(param_points)  # Shape (N, 2, 2)
# uX_in_chart = jnp.einsum('nij,nj->ni', jac_chart, uX)
# uY_in_chart = jnp.einsum('nij,nj->ni', jac_chart, uY)
# uX_in_chart /= jnp.linalg.norm(uX_in_chart, axis=1, keepdims=True)
# uY_in_chart /= jnp.linalg.norm(uY_in_chart, axis=1, keepdims=True)
# plot_tools.plot_vector_field_in_chart(
#     ax, points_in_chart, uX_in_chart, vector_scale=0.25, colormap="plasma", label=r"$X$", inset_position=(0.02, 0.95),
#     inset_size=(0.6, 0.2), inset_orientation="horizontal", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black",
#     label_fontsize=14, arrow_style=dict(head_width=0.08, head_length=0.1, linewidth=1.2, alpha=1.0)
# )
# plot_tools.plot_vector_field_in_chart(
#     ax, points_in_chart, uY_in_chart, vector_scale=0.25, colormap="Greens", label=r"$Y$", inset_position=(0.87, 0.95),
#     inset_size=(0.6, 0.2), inset_orientation="horizontal", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black",
#     label_fontsize=14, arrow_style=dict(head_width=0.08, head_length=0.1, linewidth=1.2, alpha=1.0)
# )

ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax),
       xlabel=r"$\hat{x}^1$", ylabel=r"$\hat{x}^2$")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
fig.savefig(os.path.join(config.save_dir, name_save_prefix + "_chart_U.pdf"))
plt.show()


# === Chart V plot ===
fig, ax = plt.subplots(figsize=(7, 7))

p_coords = chart_V.map_to_chart(p_param)
p_coords = p_coords[0] if p_coords.ndim == 2 and p_coords.shape[0] == 1 else p_coords
ax.scatter(*p_coords, color="black", s=60, zorder=5, label=r"$p$")

# plot_charts.plot_parametric_region_in_chart_coordinates(
#     ax=ax, chart=chart_V, param_space_data=chart_U.boundary_curve, color="red", linestyle="dashed",
#     linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{U})$", label_position="right",
#     label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
#     mark_boundary_points=False, boundary_marker_color="red", boundary_marker_size=8
# )
# plot_charts.plot_parametric_region_in_chart_coordinates(
#     ax=ax, chart=chart_V, param_space_data=chart_V.boundary_curve, color="blue", linestyle="dashed",
#     linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{V})$", label_position="right",
#     label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
#     mark_boundary_points=False, boundary_marker_color="blue", boundary_marker_size=8
# )
# plot_charts.plot_parametric_region_in_chart_coordinates(
#     ax=ax, chart=chart_V, param_space_data=chart_U_cap_V_X.boundary_curve, color="purple", linestyle="dotted",
#     linewidth=3.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{U} \cap \mathcal{V})$", label_position="center",
#     label_fontsize=16, tessellated=False, simplices=None, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
#     mark_boundary_points=False, boundary_marker_color="purple", boundary_marker_size=8
# )
param_points = np.array(chart_U.sample_region_in_param_space(n_points=2**11))
tri = Delaunay(param_points)
plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_V, param_space_data=param_points, color="red", linestyle="dashed",
    linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{U})$", label_position="right",
    label_fontsize=16, tessellated=True, simplices=tri.simplices, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=True, boundary_marker_color="red", boundary_marker_size=8
)
param_points = np.array(chart_V.sample_region_in_param_space(n_points=2**11))
tri = Delaunay(param_points)
plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_V, param_space_data=param_points, color="blue", linestyle="dashed",
    linewidth=1.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{V})$", label_position="right",
    label_fontsize=16, tessellated=True, simplices=tri.simplices, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=True, boundary_marker_color="blue", boundary_marker_size=8
)
param_points = np.array(chart_U_cap_V_X.sample_region_in_param_space(n_points=2**11))
tri = Delaunay(param_points)
plot_charts.plot_parametric_region_in_chart_coordinates(
    ax=ax, chart=chart_V, param_space_data=param_points, color="purple", linestyle="dotted",
    linewidth=3.5, alpha=0.3, edgecolor="none", label=r"$y(\mathcal{U} \cap \mathcal{V})$", label_position="center",
    label_fontsize=16, tessellated=True, simplices=tri.simplices, tessellation_edgecolor="gray", tessellation_linewidth=0.1,
    mark_boundary_points=True, boundary_marker_color="purple", boundary_marker_size=8
)


plot_tools.plot_curve_in_chart(
    ax, chart=chart_V, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
    color="gold", linewidth=2.0, label=r"$\gamma$", label_position="end", label_fontsize=20, label_offset=(0.1, 0.1)
)
plot_tools.plot_curve_in_chart(
    ax, chart=chart_V, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
    color="limegreen", linewidth=2.0, label=r"$\delta$", label_position="end", label_fontsize=20, label_offset=(0.1, 0.1)
)

# plot_tools.plot_chart_components_of_curve_tangent(
#     ax, chart=chart_V, curve=curve_gamma, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
#     step=30, scale=0.3, color="gold", linewidth=1.8, head_width=0.1, head_length=0.2, zorder=5, method="finite_difference"
# )
# plot_tools.plot_chart_components_of_curve_tangent(
#     ax, chart=chart_V, curve=curve_delta, lambda_range=(-jnp.pi, jnp.pi), n_points=300,
#     step=30, scale=0.3, color="limegreen", linewidth=1.8, head_width=0.1, head_length=0.2, zorder=5, method="finite_difference"
# )

# plot_tools.plot_function_in_chart(
#     ax=ax, chart=chart_V, function=function_f, param_sampling_bounds=((xmin, xmax), (ymin, ymax)),
#     chart_xlim=(xmin, xmax), chart_ylim=(ymin, ymax), resolution=96, cmap="viridis", alpha=0.5, label_function="Scalar field $f$"
# )

# plot_tools.plot_colored_curve_directional_derivative_in_chart(
#     ax=ax, chart=chart_V, curve=curve_gamma, scalar_function=function_f, cmap="PiYG", label=r"$\gamma$", label_position="end",
#     inset_colorbar=True, inset_position=(0.02, 0.05), inset_size=(0.6, 0.2), colorbar_orientation="horizontal",
#     label_colorbar=r"$\left(\gamma \circ f\right)'$", label_colorbar_position="top"
# )

# plot_tools.plot_colored_curve_directional_derivative_in_chart(
#     ax=ax, chart=chart_V, curve=curve_delta, scalar_function=function_f, cmap="bwr", label=r"$\delta$", label_position="end",
#     inset_colorbar=True, inset_position=(0.85, 0.05), inset_size=(0.6, 0.2), colorbar_orientation="horizontal",
#     label_colorbar=r"$\left(\delta \circ f\right)'$", label_colorbar_position="top"
# )

# xv, yv = jnp.linspace(xmin, xmax, grid_resolution_field), jnp.linspace(ymin, ymax, grid_resolution_field)
# param_points = jnp.stack(jnp.meshgrid(xv, yv), axis=-1).reshape(-1, 2)
# points_in_chart = chart_V.map_to_chart(param_points)
# uX = field_X.evaluate_in_param_space(param_points)
# uY = field_Y.evaluate_in_param_space(param_points)
# def chart_map_fn(param):
#     return chart_V.map_to_chart(param)
# jac_chart = jax.vmap(jax.jacrev(chart_map_fn))(param_points)  # Shape (N, 2, 2)
# uX_in_chart = jnp.einsum('nij,nj->ni', jac_chart, uX)
# uY_in_chart = jnp.einsum('nij,nj->ni', jac_chart, uY)
# uX_in_chart /= jnp.linalg.norm(uX_in_chart, axis=1, keepdims=True)
# uY_in_chart /= jnp.linalg.norm(uY_in_chart, axis=1, keepdims=True)
# plot_tools.plot_vector_field_in_chart(
#     ax, points_in_chart, uX_in_chart, vector_scale=0.25, colormap="YlOrBr", label=r"$X$", inset_position=(0.02, 0.95),
#     inset_size=(0.6, 0.2), inset_orientation="horizontal", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black",
#     label_fontsize=14, arrow_style=dict(head_width=0.08, head_length=0.1, linewidth=1.2, alpha=1.0)
# )
# plot_tools.plot_vector_field_in_chart(
#     ax, points_in_chart, uY_in_chart, vector_scale=0.25, colormap="Greens", label=r"$Y$", inset_position=(0.87, 0.95),
#     inset_size=(0.6, 0.2), inset_orientation="horizontal", inset_box_alpha=0.9, inset_box_color="white", inset_border_color="black",
#     label_fontsize=14, arrow_style=dict(head_width=0.08, head_length=0.1, linewidth=1.2, alpha=1.0)
# )

ax.set(xlim=(-0.2, 5.5), ylim=(-np.pi-0.2, np.pi+0.2),
       xlabel=r"$\hat{y}^1$", ylabel=r"$\hat{y}^2$")
ax.axvspan(-1, 0, color='grey', alpha=0.2, hatch='//')
ax.axhspan(np.pi, np.pi+1, color='grey', alpha=0.2, hatch='//')
ax.axhspan(-np.pi-1, -np.pi, color='grey', alpha=0.2, hatch='//')
ax.axhline(np.pi, color='gray', ls='--', lw=1)
ax.axhline(-np.pi, color='gray', ls='--', lw=1)
ax.axvline(0, color='gray', ls='--', lw=1)
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
fig.savefig(os.path.join(config.save_dir, name_save_prefix + "_chart_V.pdf"))
plt.show()

In [ ]:
# # === Prepare curve data (reduced resolution) ===
# lambda_vals = jnp.linspace(-jnp.pi, jnp.pi, 50)  # ⬅️ Reduced from 500 to 200
# curve_gamma_pts = manifold.embed(curve_gamma.evaluate_in_param_space(lambda_vals))
# curve_delta_pts = manifold.embed(curve_delta.evaluate_in_param_space(lambda_vals))

# # === Create figure ===
# xmin, xmax = jnp.array([-jnp.pi, jnp.pi]) * jnp.pi / 3
# ymin, ymax = jnp.array([-jnp.pi, jnp.pi]) * jnp.pi / 3

# fig, ax = plot_tools.create_3d_axis_for_manifold(
#     xlim=(xmin, xmax),
#     ylim=(ymin, ymax),
#     zlim=(0, 2.5),
#     axis_labels=None
# )

# # === Plot static surface (reduced resolution) ===
# plot_tools.plot_manifold_surface(
#     ax=ax,
#     surface_manifold=manifold,
#     resolution=64,  # ⬅️ Reduced from 128 to 64
#     xmin=xmin,
#     xmax=xmax,
#     ymin=ymin,
#     ymax=ymax,
#     color="skyblue",
#     alpha=0.2,
#     edgecolor="none",
#     label=r"$\left(\mathcal{M}, \mathcal{O}, \mathcal{A}\right)$",
#     label_position="right",
#     label_fontsize=16
# )

# # === Plot chart regions ===
# plot_tools.plot_chart_region_manifold(
#     ax=ax, surface_manifold=surface_manifold, chart=chart_U,
#     color='red', linestyle='dashed', linewidth=1.5, fill_surface=True,
#     alpha=0.15, edgecolor='red', label=r"$\mathcal{U}$", label_position="bottom"
# )

# plot_tools.plot_chart_region_manifold(
#     ax=ax, surface_manifold=surface_manifold, chart=chart_V,
#     color='blue', linestyle='dashed', linewidth=1.5, fill_surface=True,
#     alpha=0.15, edgecolor='blue', label=r"$\mathcal{V}$", label_position="bottom"
# )

# plot_tools.plot_chart_intersection(
#     ax=ax, surface_manifold=surface_manifold, chart1=chart_U, chart2=chart_V,
#     color='purple', linestyle='dotted', linewidth=3.5, fill_surface=True,
#     alpha=0.2, edgecolor='purple', label=None, label_position="center"
# )

# # === Plot point and tangent plane ===
# p_xyz = surface_manifold.embed(jnp.array([[p_x, p_y]]))[0]
# ax.scatter(p_xyz[0], p_xyz[1], p_xyz[2], color="k", s=60)

# p_xyz, t1, t2 = aux.compute_tangent_plane(surface_manifold, p_param)
# plot_tools.plot_tangent_plane(ax, p_xyz, t1, t2, size=1.8, color="purple", alpha=0.35)

# # === Initialize curves ===
# line_gamma, = ax.plot([], [], [], color="gold", linewidth=2.5, label=r"$\gamma$")
# line_delta, = ax.plot([], [], [], color="limegreen", linewidth=2.5, label=r"$\delta$")

# # === Animation update ===
# def update(frame):
#     ax.view_init(elev=50, azim=-60 + frame * 1.0)  # ⬅️ Slightly faster rotation

#     line_gamma.set_data(curve_gamma_pts[:frame, 0], curve_gamma_pts[:frame, 1])
#     line_gamma.set_3d_properties(curve_gamma_pts[:frame, 2])

#     line_delta.set_data(curve_delta_pts[:frame, 0], curve_delta_pts[:frame, 1])
#     line_delta.set_3d_properties(curve_delta_pts[:frame, 2])

#     return line_gamma, line_delta

# # === Create animation ===
# frames = len(lambda_vals)  # ⬅️ 200 frames instead of 500
# ani = animation.FuncAnimation(
#     fig, update, frames=frames, interval=40, blit=False
# )

# # === Save animation ===
# ani.save("manifold_animation.mp4", fps=30, bitrate=1800)

# plt.show()